# PCNSL Olink — transcriptomics

## Setup

In [ ]:
import os, math, warnings, shutil
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')
for k in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[k] = '1'
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from IPython.display import SVG, display

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 250, 'font.size': 10})

ROOT         = Path('outputs')
PANEL_V2     = Path('outputs/figures_v2')
OUTROOT      = Path('outputs/supp_paired_rna')
for _p in (ROOT, PANEL_V2, OUTROOT):
    _p.mkdir(parents=True, exist_ok=True)

def _preview(fig_path, *, width=1000):
    display(SVG(filename=str(fig_path)))

def save_panel(fig, slug, *, copy=True, preview=True, close=True, svg_dpi=600):
    svg = PANEL_V2 / f'{slug}.svg'
    fig.savefig(svg, dpi=svg_dpi, bbox_inches='tight')
    if preview:
        _preview(svg)
    if close:
        plt.close(fig)
    return svg

SIGNATURE_DEFS = {
    'IFN_Chemokine': ['CXCL9', 'CXCL10', 'IFNG', 'CCL19', 'CCL20', 'CCL3', 'CCL4'],
    'Checkpoint_Cytokine': ['IL10', 'IL6', 'PDCD1', 'PDCD1LG2', 'LAG3', 'CD86', 'TNF'],
    'Innate_Inflammatory': ['IL1B', 'IL6', 'TNF', 'CCL20', 'CCL3', 'CCL4', 'FASLG'],
    'Integrated_Immune': ['IL10', 'IL6', 'IL1B', 'CCL19', 'PDCD1LG2', 'PDCD1',
                          'CXCL9', 'CXCL10', 'IFNG', 'LAG3', 'CD86', 'TNF',
                          'CCL20', 'CCL3', 'CCL4', 'TNFRSF13C', 'FASLG'],
}
SIG_DISPLAY = {'IFN_Chemokine': 'IFN/Chemokine', 'Checkpoint_Cytokine': 'Checkpoint/Cytokine',
               'Innate_Inflammatory': 'Innate/Inflammatory', 'Integrated_Immune': 'Integrated Immune'}
ALL_SIG_GENES = sorted(set(g for gs in SIGNATURE_DEFS.values() for g in gs))

IMMUNE_CELL_SIGNATURES = {
    'CD8+ T cells': ['CD8A', 'CD8B', 'GZMA', 'GZMB', 'PRF1', 'IFNG', 'NKG7', 'EOMES'],
    'CD4+ T cells': ['CD4', 'IL7R', 'TCF7', 'LEF1', 'FOXP3', 'IL2RA'],
    'NK cells': ['KLRD1', 'KLRK1', 'NCR1', 'NCR3', 'GNLY', 'NKG7', 'KLRF1'],
    'B cells': ['CD19', 'MS4A1', 'CD79A', 'CD79B', 'PAX5', 'BANK1', 'BLK'],
    'Macrophages': ['CD68', 'CD14', 'CSF1R', 'MSR1', 'MARCO', 'CD163', 'MRC1'],
    'Dendritic cells': ['ITGAX', 'CLEC10A', 'CD1C', 'FCER1A', 'BATF3', 'IRF8'],
    'Tregs': ['FOXP3', 'IL2RA', 'CTLA4', 'TNFRSF18', 'IKZF2'],
    'Cytotoxic score': ['GZMA', 'GZMB', 'GZMH', 'PRF1', 'GNLY', 'FASLG', 'NKG7'],
    'Inflammatory score': ['IL1B', 'IL6', 'TNF', 'CCL3', 'CCL4', 'CXCL8', 'IL18'],
    'Checkpoint exhaustion': ['PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'CTLA4', 'PDCD1LG2', 'CD274']}

## 1. Data loading and plasma immune score

Loads the proteomics replicates CSV, computes signature scores, reproduces the
consensus 2-means split (Inflamed vs Non-inflamed), then loads the FF and FFPE RNA-seq
tables and matches patients by `omics_ID`.


In [ ]:
prot_raw = pd.read_csv('data/pcnsl/proteomics/data_PCNSL_survival_olink_replicates_r0955.csv')
META_COLS = ['patient_id', 'MRI', 'omics_ID', 'sex_completed', 'age_years', 'cohort',
             'Karnofsky_status', 'Karnofsky_group', 'estimated_OS_months',
             'estimated_OS_event', 'estimated_PFS_months', 'estimated_PFS_event']
protein_cols_all = [c for c in prot_raw.columns if c not in META_COLS]

KNOWN_COMPOSITES = {'DEFB103A_DEFB103B': 'DEFB103A', 'IL12A_IL12B': 'IL12A', 'EBI3_IL27': 'EBI3',
                    'KIR2DL2_KIR2DL3': 'KIR2DL2', 'MICA_MICB': 'MICA', 'NCF1C_NCF1': 'NCF1',
                    'RAET1L_ULBP2': 'RAET1L'}
olink_to_gene = {c: KNOWN_COMPOSITES.get(c, c) for c in protein_cols_all}
gene_to_olink = {v: k for k, v in olink_to_gene.items()}

dc = prot_raw.copy()
for c in protein_cols_all:
    dc[c] = pd.to_numeric(dc[c], errors='coerce')

for sig_name, sig_genes in SIGNATURE_DEFS.items():
    keep = [c for c in sig_genes if c in protein_cols_all]
    dc[sig_name + '_score'] = dc[keep].mean(axis=1)

cohort_col = dc['cohort'].astype(str).str.lower()
train_mask = cohort_col.eq('training').values
test_mask = cohort_col.eq('validation').values

sig_roots = ['IFN_Chemokine', 'Checkpoint_Cytokine', 'Innate_Inflammatory', 'Integrated_Immune']
cluster_protein_cols = [c for c in dc.columns if c not in META_COLS
                        and not any(c.startswith(s) for s in sig_roots)]

X_all = dc[cluster_protein_cols].to_numpy(float)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_all[train_mask])
X_test = scaler.transform(X_all[test_mask])
X_train = X_train - np.median(X_train, axis=1, keepdims=True)
X_test = X_test - np.median(X_test, axis=1, keepdims=True)

final_model = KMeans(n_clusters=2, n_init=100, random_state=42, max_iter=300, algorithm='lloyd').fit(X_train)
train_labels = final_model.labels_
test_labels = final_model.predict(X_test)

train_df_tmp = dc.loc[train_mask].copy()
cluster_means = train_df_tmp.groupby(train_labels)['Integrated_Immune_score'].mean().sort_values(ascending=False)
inflamed_cluster = int(cluster_means.index[0])

full_labels = np.empty(len(dc), dtype=int)
full_labels[train_mask] = train_labels
full_labels[test_mask] = test_labels
dc['serum_group'] = np.where(full_labels == inflamed_cluster, 'Inflamed', 'Non-inflamed')

n_infl = (dc['serum_group'] == 'Inflamed').sum()
n_non = (dc['serum_group'] == 'Non-inflamed').sum()
print(f'Full cohort: {n_infl} Inflamed, {n_non} Non-inflamed (total {len(dc)})')

ff_vst = pd.read_csv('data/pcnsl/transcriptomics/FF_n10_vst.tab', sep='\t', index_col=0)
ffpe_vst = pd.read_csv('data/pcnsl/transcriptomics/FFPE_n8_Degnorm_VST.tab', sep='\t', index_col=0)
ff_tpm = pd.read_csv('data/pcnsl/transcriptomics/FF_n10_TPM.tab', sep='\t', index_col=0)
ffpe_tpm = pd.read_csv('data/pcnsl/transcriptomics/FFPE_n8_Degnorm_TPM.tab', sep='\t', index_col=0)

dc['omics_ID_str'] = dc['omics_ID'].astype(str).str.strip()
matched = dc[dc['omics_ID_str'].isin(set(ff_vst.columns) | set(ffpe_vst.columns))].copy()
matched['tissue'] = matched['omics_ID_str'].apply(lambda x: 'FF' if x.startswith('CNSL') else 'FFPE')

matched_ff = matched[matched['tissue'] == 'FF'].copy()
matched_ffpe = matched[matched['tissue'] == 'FFPE'].copy()

print(f'\nMatched: {len(matched_ff)} FF + {len(matched_ffpe)} FFPE = {len(matched)} total')
for tissue, sub in [('FF', matched_ff), ('FFPE', matched_ffpe)]:
    ni = (sub['serum_group'] == 'Inflamed').sum()
    nn = (sub['serum_group'] == 'Non-inflamed').sum()
    score_range = sub['Integrated_Immune_score'].agg(['min', 'max'])
    print(f'  {tissue}: {ni} Inflamed, {nn} Non-inflamed, '
          f'Integrated Immune score range: [{score_range["min"]:.2f}, {score_range["max"]:.2f}]')

## 2. Build matched matrices for signature genes

In [ ]:
def build_matched_matrix(prot_df, rna_df, gene_list):
    ids = prot_df['omics_ID_str'].tolist()
    prot_mat = np.full((len(ids), len(gene_list)), np.nan)
    rna_mat = np.full((len(ids), len(gene_list)), np.nan)
    for i, (_, row) in enumerate(prot_df.iterrows()):
        sid = row['omics_ID_str']
        for j, gene in enumerate(gene_list):
            ocol = gene_to_olink.get(gene, gene)
            if ocol in prot_df.columns:
                try: prot_mat[i, j] = float(row[ocol])
                except: pass
            if gene in rna_df.index and sid in rna_df.columns:
                try: rna_mat[i, j] = float(rna_df.loc[gene, sid])
                except: pass
    return prot_mat, rna_mat, ids

olink_genes = set(olink_to_gene.values())
sig_genes_ff = sorted(set(ALL_SIG_GENES) & olink_genes & set(ff_vst.index))
sig_genes_ffpe = sorted(set(ALL_SIG_GENES) & olink_genes & set(ffpe_vst.index))
all_genes_ff = sorted(olink_genes & set(ff_vst.index))
all_genes_ffpe = sorted(olink_genes & set(ffpe_vst.index))

prot_ff_mat, rna_ff_mat, ff_ids = build_matched_matrix(matched_ff, ff_vst, sig_genes_ff)
prot_ffpe_mat, rna_ffpe_mat, ffpe_ids = build_matched_matrix(matched_ffpe, ffpe_vst, sig_genes_ffpe)

_, rna_ff_all, _ = build_matched_matrix(matched_ff, ff_vst, all_genes_ff)
_, rna_ffpe_all, _ = build_matched_matrix(matched_ffpe, ffpe_vst, all_genes_ffpe)

ff_groups = matched_ff['serum_group'].values
ffpe_groups = matched_ffpe['serum_group'].values
ff_plasma_score = matched_ff['Integrated_Immune_score'].values
ffpe_plasma_score = matched_ffpe['Integrated_Immune_score'].values

print(f'Signature genes matched: {len(sig_genes_ff)} FF, {len(sig_genes_ffpe)} FFPE')


## 3. per-gene plasma NPX × tumour VST correlations


In [ ]:
sig_corr_rows = []
for tissue, pmat, rmat, genes in [
    ('FF', prot_ff_mat, rna_ff_mat, sig_genes_ff),
    ('FFPE', prot_ffpe_mat, rna_ffpe_mat, sig_genes_ffpe),
]:
    for j, gene in enumerate(genes):
        px, rx = pmat[:, j], rmat[:, j]
        v = np.isfinite(px) & np.isfinite(rx)
        if v.sum() >= 4:
            rho, p = stats.spearmanr(px[v], rx[v])
            sig_corr_rows.append({'gene': gene, 'tissue': tissue, 'rho': rho, 'pvalue': p, 'n': int(v.sum())})

sig_corr_df = pd.DataFrame(sig_corr_rows)
print('Analysis 1 — per-gene signature correlations:')
for tissue in ['FF', 'FFPE']:
    sub = sig_corr_df[sig_corr_df['tissue'] == tissue]
    n_sig = (sub['pvalue'] < 0.05).sum()
    print(f'  {tissue}: median rho = {sub["rho"].median():.3f}, P<0.05: {n_sig}/{len(sub)}')

### Panel a

In [ ]:
palette = {'Inflamed': '#c44e52', 'Non-inflamed': '#4c78a8'}
gene_order = ['IL6', 'IL1B', 'TNF', 'CXCL9', 'CXCL10', 'IFNG',
              'CCL3', 'CCL4', 'CCL19', 'CCL20', 'IL10', 'FASLG',
              'PDCD1', 'PDCD1LG2', 'LAG3', 'CD86', 'TNFRSF13C']
genes_to_plot = [g for g in gene_order if g in sig_genes_ff]
n_genes = len(genes_to_plot)
ncols = 6
nrows = math.ceil(n_genes / ncols)
fig_A = plt.figure(figsize=(15, 8))
gs_a = fig_A.add_gridspec(nrows, ncols, hspace=0.7, wspace=0.4)
for k, gene in enumerate(genes_to_plot):
    ax = fig_A.add_subplot(gs_a[k // ncols, k % ncols])
    gi = sig_genes_ff.index(gene)
    px, rx = prot_ff_mat[:, gi], rna_ff_mat[:, gi]
    v = np.isfinite(px) & np.isfinite(rx)
    for grp, color in palette.items():
        m = v & (ff_groups == grp)
        if m.any():
            ax.scatter(px[m], rx[m], c=color, s=35, alpha=0.85, edgecolors='black', linewidth=0.4, label=grp, zorder=3)
    if v.sum() >= 4:
        rho, pv = stats.spearmanr(px[v], rx[v])
        z = np.polyfit(px[v], rx[v], 1)
        xl = np.linspace(px[v].min(), px[v].max(), 30)
        ax.plot(xl, np.polyval(z, xl), 'k--', alpha=0.3, linewidth=0.8)
        ax.set_title(f'{gene}\nrho={rho:.2f}, P={pv:.2g}', fontsize=8)
    else:
        ax.set_title(gene, fontsize=8)
    ax.set_xlabel('Plasma NPX', fontsize=6)
    ax.set_ylabel('Tumor VST', fontsize=6)
    ax.tick_params(labelsize=5)
    if k == 0:
        ax.legend(fontsize=5, loc='upper left', markerscale=0.7, handletextpad=0.3)
for k in range(n_genes, nrows * ncols):
    fig_A.add_subplot(gs_a[k // ncols, k % ncols]).axis('off')
fig_A.suptitle('A. Plasma protein vs tumor mRNA for immune signature genes (FF, n=10)',
               fontsize=12, fontweight='bold')
fig_A.tight_layout(rect=[0, 0, 1, 0.95])
save_panel(fig_A, 'supp_paired_rna_aux_panelA_gene_scatter', copy=False, preview=True)


## signature-level concordance with bootstrap CIs


In [ ]:
def plasma_sig_score(prot_df, sig_genes):
    cols = [c for c in sig_genes if c in protein_cols_all]
    vals = prot_df[cols].apply(pd.to_numeric, errors='coerce')
    raw = vals.mean(axis=1)
    mu, sd = raw.mean(), raw.std()
    return ((raw - mu) / sd if sd > 0 else raw - mu).values

def rna_sig_score(rna_mat, gene_list, sig_genes):
    idx = [gene_list.index(g) for g in sig_genes if g in gene_list]
    if not idx:
        return np.full(rna_mat.shape[0], np.nan)
    raw = np.nanmean(rna_mat[:, idx], axis=1)
    mu, sd = np.nanmean(raw), np.nanstd(raw)
    return (raw - mu) / sd if sd > 0 else raw - mu

concordance_rows = []
for sig_name, sig_genes in SIGNATURE_DEFS.items():
    for tissue, prot_sub, rna_mat, gene_list in [
        ('FF', matched_ff, rna_ff_all, all_genes_ff),
        ('FFPE', matched_ffpe, rna_ffpe_all, all_genes_ffpe),
    ]:
        ps = plasma_sig_score(prot_sub, sig_genes)
        rs = rna_sig_score(rna_mat, gene_list, sig_genes)
        v = np.isfinite(ps) & np.isfinite(rs)
        if v.sum() >= 4:
            rho, p = stats.spearmanr(ps[v], rs[v])
            concordance_rows.append({'signature': sig_name, 'tissue': tissue,
                                      'rho': rho, 'pvalue': p, 'n': int(v.sum())})

conc_df = pd.DataFrame(concordance_rows)
print('Analysis 2 — signature-level concordance (plasma vs tumor):')
print(conc_df.to_string(index=False))


### signature concordance forest plot (95% bootstrap CI)

In [ ]:
tissue_colors = {'FF': '#5DADE2', 'FFPE': '#E67E22'}
rng = np.random.default_rng(42)
fig_B, ax_forest = plt.subplots(figsize=(6, 5))
y_pos = 0
y_labels, y_ticks = [], []
for sig_name in SIGNATURE_DEFS:
    for tissue, prot_sub, rna_mat, gene_list, color in [
        ('FF', matched_ff, rna_ff_all, all_genes_ff, tissue_colors['FF']),
        ('FFPE', matched_ffpe, rna_ffpe_all, all_genes_ffpe, tissue_colors['FFPE']),
    ]:
        row = conc_df[(conc_df['signature'] == sig_name) & (conc_df['tissue'] == tissue)]
        if len(row) == 0:
            y_pos += 1; continue
        rho_obs = row['rho'].iloc[0]
        ps = plasma_sig_score(prot_sub, SIGNATURE_DEFS[sig_name])
        rs = rna_sig_score(rna_mat, gene_list, SIGNATURE_DEFS[sig_name])
        vv = np.isfinite(ps) & np.isfinite(rs)
        ps_v, rs_v = ps[vv], rs[vv]
        boot = [stats.spearmanr(ps_v[rng.choice(len(ps_v), len(ps_v), replace=True)],
                                 rs_v[rng.choice(len(rs_v), len(rs_v), replace=True)])[0]
                for _ in range(1000)]
        ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
        ax_forest.plot([ci_lo, ci_hi], [y_pos, y_pos], color=color, linewidth=2.5)
        ax_forest.scatter([rho_obs], [y_pos], color=color, s=70, zorder=5,
                          edgecolors='black', linewidth=0.5)
        y_labels.append(f'{SIG_DISPLAY[sig_name]} ({tissue})')
        y_ticks.append(y_pos)
        y_pos += 1
ax_forest.set_yticks(y_ticks)
ax_forest.set_yticklabels(y_labels, fontsize=8)
ax_forest.axvline(0, color='grey', linewidth=0.5, linestyle='--')
ax_forest.set_xlabel('Spearman rho (plasma vs tumor)', fontsize=9)
ax_forest.set_title('B. Signature concordance (95% bootstrap CI)', fontsize=10, fontweight='bold')
ax_forest.invert_yaxis()
ax_forest.set_xlim(-1.3, 1.3)
fig_B.tight_layout()
save_panel(fig_B, 'supp_paired_rna_aux_panelB_signature_forest', copy=False, preview=True)
print('Saved aux panel B (signature concordance forest)')


### IL10 plasma × tumour scatter (FF and FFPE)

In [ ]:
for col_idx, (tissue, pmat, rmat, genes, groups, n_label, letter) in enumerate([
    ('FF', prot_ff_mat, rna_ff_mat, sig_genes_ff, ff_groups, 'n=10', 'C'),
    ('FFPE', prot_ffpe_mat, rna_ffpe_mat, sig_genes_ffpe, ffpe_groups, 'n=8', 'D'),
]):
    fig_, ax = plt.subplots(figsize=(5, 4))
    highlight_gene = 'IL10'
    if highlight_gene in genes:
        gi = genes.index(highlight_gene)
        px, rx = pmat[:, gi], rmat[:, gi]
        v = np.isfinite(px) & np.isfinite(rx)
        for grp, color in palette.items():
            m = v & (groups == grp)
            if m.any():
                ax.scatter(px[m], rx[m], c=color, s=60, alpha=0.85,
                           edgecolors='black', linewidth=0.5, label=grp, zorder=3)
        if v.sum() >= 4:
            rho, pv = stats.spearmanr(px[v], rx[v])
            z = np.polyfit(px[v], rx[v], 1)
            xl = np.linspace(px[v].min(), px[v].max(), 30)
            ax.plot(xl, np.polyval(z, xl), 'k--', alpha=0.4, linewidth=1)
            ax.text(0.05, 0.95, f'rho = {rho:.2f}\nP = {pv:.3g}',
                    transform=ax.transAxes, ha='left', va='top', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
        ax.set_xlabel(f'Plasma {highlight_gene} NPX', fontsize=9)
        ax.set_ylabel(f'Tumor {highlight_gene} VST', fontsize=9)
        ax.set_title(f'{letter}. {highlight_gene} ({tissue}, {n_label})', fontsize=10, fontweight='bold')
        ax.legend(fontsize=8)
    fig_.tight_layout()
    save_panel(fig_, f'supp_paired_rna_aux_panel{letter}_IL10_{tissue}', copy=False, preview=True)
print('Saved aux panels C and D (IL10 scatter)')


## ssGSEA tumour immune cells × continuous plasma score


In [ ]:
def ssgsea_proper(expr_mat, gene_names, gene_set, alpha=0.25):
    present = set(gene_names.index(g) for g in gene_set if g in gene_names)
    if len(present) < 2:
        return np.full(expr_mat.shape[0], np.nan)
    mask = np.array([i in present for i in range(len(gene_names))])
    scores = []
    for i in range(expr_mat.shape[0]):
        row = expr_mat[i, :]
        finite = np.isfinite(row)
        if finite.sum() < 10:
            scores.append(np.nan); continue
        order = np.argsort(-row)
        sm = mask[order]
        rw = np.abs(rankdata(row))[order] ** alpha
        hs = (rw * sm).sum()
        if hs == 0:
            scores.append(0.0); continue
        hit = np.cumsum(rw * sm) / hs
        miss = np.cumsum(~sm) / (~sm).sum()
        scores.append((hit - miss).sum())
    return np.array(scores)

ff_tpm_genes = list(ff_tpm.index)
ff_tpm_mat = np.log2(ff_tpm[matched_ff['omics_ID_str'].tolist()].T.values.astype(float) + 1)
ffpe_tpm_genes = list(ffpe_tpm.index)
ffpe_tpm_mat = np.log2(ffpe_tpm[matched_ffpe['omics_ID_str'].tolist()].T.values.astype(float) + 1)

cell_types = list(IMMUNE_CELL_SIGNATURES.keys())
deconv_ff = pd.DataFrame(index=ff_ids, columns=cell_types, dtype=float)
deconv_ffpe = pd.DataFrame(index=ffpe_ids, columns=cell_types, dtype=float)
for ct, markers in IMMUNE_CELL_SIGNATURES.items():
    sc = ssgsea_proper(ff_tpm_mat, ff_tpm_genes, markers)
    mu, sd = np.nanmean(sc), np.nanstd(sc)
    deconv_ff[ct] = (sc - mu) / sd if sd > 0 else sc - mu
    sc = ssgsea_proper(ffpe_tpm_mat, ffpe_tpm_genes, markers)
    mu, sd = np.nanmean(sc), np.nanstd(sc)
    deconv_ffpe[ct] = (sc - mu) / sd if sd > 0 else sc - mu

deconv_corr_rows = []
for tissue, deconv, plasma_sc in [
    ('FF', deconv_ff, ff_plasma_score),
    ('FFPE', deconv_ffpe, ffpe_plasma_score),
]:
    for ct in cell_types:
        tumor_sc = deconv[ct].values.astype(float)
        v = np.isfinite(tumor_sc) & np.isfinite(plasma_sc)
        if v.sum() >= 4:
            rho, p = stats.spearmanr(plasma_sc[v], tumor_sc[v])
            deconv_corr_rows.append({'cell_type': ct, 'tissue': tissue, 'rho': rho, 'pvalue': p, 'n': int(v.sum())})
deconv_corr_df = pd.DataFrame(deconv_corr_rows)
print('Analysis 3 — tumour immune cells vs plasma Integrated Immune score:')
print(deconv_corr_df.to_string(index=False))


### 10 cell-type scatter grid vs plasma Integrated Immune score

In [ ]:
from matplotlib.lines import Line2D as _Line2D
from matplotlib.patches import Patch as _Patch
fig_E = plt.figure(figsize=(15, 7))
gs_c = fig_E.add_gridspec(2, 5, hspace=0.55, wspace=0.5)
for j, ct in enumerate(cell_types):
    ax = fig_E.add_subplot(gs_c[j // 5, j % 5])
    tumor_ff = deconv_ff[ct].values.astype(float)
    tumor_ffpe = deconv_ffpe[ct].values.astype(float)
    v_ff = np.isfinite(ff_plasma_score) & np.isfinite(tumor_ff)
    if v_ff.sum() >= 3:
        for grp, color in palette.items():
            m = v_ff & (ff_groups == grp)
            if m.any():
                ax.scatter(ff_plasma_score[m], tumor_ff[m], c=color, s=40, alpha=0.8,
                           edgecolors='black', linewidth=0.4, marker='o', zorder=3)
    v_ffpe = np.isfinite(ffpe_plasma_score) & np.isfinite(tumor_ffpe)
    if v_ffpe.sum() >= 3:
        for grp, color in palette.items():
            m = v_ffpe & (ffpe_groups == grp)
            if m.any():
                ax.scatter(ffpe_plasma_score[m], tumor_ffpe[m], c=color, s=30, alpha=0.5,
                           edgecolors='black', linewidth=0.3, marker='s', zorder=2)
    if v_ff.sum() >= 4:
        rho, pv = stats.spearmanr(ff_plasma_score[v_ff], tumor_ff[v_ff])
        z = np.polyfit(ff_plasma_score[v_ff], tumor_ff[v_ff], 1)
        xl = np.linspace(ff_plasma_score[v_ff].min(), ff_plasma_score[v_ff].max(), 20)
        ax.plot(xl, np.polyval(z, xl), '-', color=tissue_colors['FF'], alpha=0.6, linewidth=1.5)
        stars = '*' if pv < 0.05 else 'dagger' if pv < 0.1 else ''
        ax.set_title(f'{ct}\nrho={rho:.2f} {stars}', fontsize=8)
    else:
        ax.set_title(ct, fontsize=8)
    ax.set_xlabel('Plasma Integrated\nImmune score', fontsize=6)
    ax.set_ylabel('ssGSEA (z)', fontsize=6)
    ax.tick_params(labelsize=5)
    if j == 0:
        legend_elements = [
            _Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=6, label='FF'),
            _Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=5, label='FFPE'),
            _Patch(facecolor=palette['Inflamed'], label='Inflamed'),
            _Patch(facecolor=palette['Non-inflamed'], label='Non-inflamed'),
        ]
        ax.legend(handles=legend_elements, fontsize=4.5, loc='upper left',
                  handletextpad=0.3, labelspacing=0.3)
fig_E.suptitle('E. Tumor immune cell composition vs plasma Integrated Immune score', fontsize=11, fontweight='bold')
fig_E.tight_layout(rect=[0, 0, 1, 0.95])
save_panel(fig_E, 'supp_paired_rna_aux_panelE_deconvolution', copy=False, preview=False)
print('Saved aux panel E (deconvolution grid)')


## S9 (paired tumour RNA vs plasma protein)

In [ ]:
union_genes = list(SIGNATURE_DEFS['Integrated_Immune'])
PLAT_COLORS = {'FF': '#5DADE2', 'FFPE': '#E67E22'}


def _paired_block(prot_df, vst_df, genes):
    P = np.full((len(prot_df), len(genes)), np.nan)
    R = np.full((len(prot_df), len(genes)), np.nan)
    for i, (_, row) in enumerate(prot_df.iterrows()):
        sid = row['omics_ID_str']
        for j, g in enumerate(genes):
            ocol = gene_to_olink.get(g, g)
            if ocol in prot_df.columns:
                try: P[i, j] = float(row[ocol])
                except Exception: pass
            if g in vst_df.index and sid in vst_df.columns:
                try: R[i, j] = float(vst_df.loc[g, sid])
                except Exception: pass
    return P, R


def _z_cols(M):
    mu = np.nanmean(M, axis=0, keepdims=True)
    sd = np.nanstd(M, axis=0, keepdims=True)
    sd = np.where(sd > 0, sd, np.nan)
    return (M - mu) / sd


P_ff,   R_ff   = _paired_block(matched_ff,   ff_vst,   union_genes)
P_ffpe, R_ffpe = _paired_block(matched_ffpe, ffpe_vst, union_genes)

Pz = np.vstack([_z_cols(P_ff), _z_cols(P_ffpe)])
Rz = np.vstack([_z_cols(R_ff), _z_cols(R_ffpe)])
platform   = np.array(['FF'] * len(P_ff) + ['FFPE'] * len(P_ffpe))
sample_ids = matched_ff['omics_ID_str'].tolist() + matched_ffpe['omics_ID_str'].tolist()
ng, ns = len(union_genes), len(sample_ids)

rho_rows = []
for j, g in enumerate(union_genes):
    x, y = Pz[:, j], Rz[:, j]
    v = np.isfinite(x) & np.isfinite(y)
    rho, p = (stats.spearmanr(x[v], y[v]) if v.sum() >= 4 else (np.nan, np.nan))
    rho_rows.append({'gene': g, 'rho_combined': rho, 'pvalue': p, 'n': int(v.sum())})
rho_combined_df = pd.DataFrame(rho_rows)
rho_combined_df.to_csv(OUTROOT / 'paired_rna_protein_combined_rho.csv', index=False)

fig = plt.figure(figsize=(14, 7))
gs = fig.add_gridspec(2, 3, height_ratios=[22, 1], width_ratios=[1.0, 1.0, 1.0],
                      wspace=0.28, hspace=0.06)
ax_rna,  ax_rna_s = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0])
ax_prot           = fig.add_subplot(gs[:, 1])
ax_rho            = fig.add_subplot(gs[0, 2])
_vmax = 2.5

def _heat(ax, Z, title):
    im = ax.imshow(Z.T, aspect='auto', cmap='RdBu_r', vmin=-_vmax, vmax=_vmax)
    ax.set_yticks(range(ng)); ax.set_yticklabels(union_genes, fontsize=7)
    ax.set_xticks([]); ax.set_title(title, fontsize=10, fontweight='bold')
    return im

im = _heat(ax_rna,  Rz, 'Tumour RNA (VST, row z)')
_  = _heat(ax_prot, Pz, 'Plasma protein (NPX, row z)')
ax_prot.set_yticklabels([])

plat_codes = np.array([0 if t == 'FF' else 1 for t in platform])[None, :]
strip_cmap = ListedColormap([PLAT_COLORS['FF'], PLAT_COLORS['FFPE']])
ax_rna_s.imshow(plat_codes, aspect='auto', cmap=strip_cmap, vmin=0, vmax=1)
ax_rna_s.set_yticks([0]); ax_rna_s.set_yticklabels(['platform'], fontsize=7); ax_rna_s.set_xticks([])

yv       = np.arange(ng)
rho_vals = rho_combined_df['rho_combined'].values
bar_cols = ['#c44e52' if (np.isfinite(r) and r > 0) else '#4c78a8' for r in rho_vals]
ax_rho.barh(yv, np.nan_to_num(rho_vals), color=bar_cols, edgecolor='black', linewidth=0.4)
for k, (r, p) in enumerate(zip(rho_vals, rho_combined_df['pvalue'].values)):
    if np.isfinite(p) and p < 0.05:
        ax_rho.text(r + (0.04 if r >= 0 else -0.04), k, '*', va='center',
                    ha='left' if r >= 0 else 'right', fontsize=11)
ax_rho.axvline(0, color='0.4', lw=0.8)
ax_rho.set_ylim(ng - 0.5, -0.5)
ax_rho.set_yticks(yv); ax_rho.set_yticklabels(union_genes, fontsize=7)
ax_rho.set_xlim(-1, 1); ax_rho.set_xlabel('Spearman ρ (plasma vs tumour)', fontsize=8)
ax_rho.set_title('Per-gene concordance (combined)', fontsize=10, fontweight='bold')
ax_rho.grid(axis='x', alpha=0.25)

cb = fig.colorbar(im, ax=ax_rho, location='left', pad=0.22, fraction=0.05)
cb.set_label('row z-score', fontsize=8); cb.ax.tick_params(labelsize=7)

fig.legend(handles=[Patch(facecolor=PLAT_COLORS['FF'],   label=f'FF (n={len(P_ff)})'),
                    Patch(facecolor=PLAT_COLORS['FFPE'], label=f'FFPE (n={len(P_ffpe)})'),
                    Patch(facecolor='#c44e52', label='positive ρ'),
                    Patch(facecolor='#4c78a8', label='negative ρ')],
           loc='lower center', ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.02))

save_panel(fig, 'supp_fig_s7_paired_rna_protein', copy=True)

_fin = rho_combined_df['rho_combined'].dropna()
print(f'Paired RNA-protein per-gene rho: median {_fin.median():.2f}, '
      f'range [{_fin.min():.2f}, {_fin.max():.2f}], '
      f"P<0.05 {int((rho_combined_df['pvalue'] < 0.05).sum())}/{len(rho_combined_df)}")

In [ ]:
gap = np.full((1, ns), np.nan)
combined = np.vstack([Rz.T, gap, Pz.T])            
sep = ng                                            

fig = plt.figure(figsize=(11, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[40, 1], width_ratios=[2.4, 1.0],
                      wspace=0.12, hspace=0.05)
ax_h, ax_s = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0])
ax_rho = fig.add_subplot(gs[0, 1])
_vmax = 2.5

cmap = plt.get_cmap('RdBu_r').copy(); cmap.set_bad('#f5f5f5')
im = ax_h.imshow(combined, aspect='auto', cmap=cmap, vmin=-_vmax, vmax=_vmax)
ax_h.grid(False); ax_h.set_xticks([])
ax_h.set_yticks(list(range(ng)) + list(range(ng + 1, 2 * ng + 1)))
ax_h.set_yticklabels(union_genes + union_genes, fontsize=7)
ax_h.axhline(sep, color='black', lw=1.0)
ax_h.set_title('Paired tumour RNA vs plasma protein (row z)', fontsize=11, fontweight='bold')

from matplotlib.transforms import blended_transform_factory
tx = blended_transform_factory(ax_h.transAxes, ax_h.transData)
ax_h.text(-0.16, (ng - 1) / 2,          'Tumour\nRNA',     transform=tx,
          rotation=90, ha='center', va='center', fontweight='bold')
ax_h.text(-0.16, ng + 1 + (ng - 1) / 2, 'Plasma\nprotein', transform=tx,
          rotation=90, ha='center', va='center', fontweight='bold')

ax_s.imshow(plat_codes, aspect='auto', cmap=strip_cmap, vmin=0, vmax=1)
ax_s.grid(False); ax_s.set_yticks([0]); ax_s.set_yticklabels(['platform'], fontsize=7)
ax_s.set_xticks([])

yv = np.arange(ng)
rho_vals = rho_combined_df['rho_combined'].values
bar_cols = ['#c44e52' if (np.isfinite(r) and r > 0) else '#4c78a8' for r in rho_vals]
ax_rho.barh(yv, np.nan_to_num(rho_vals), color=bar_cols, edgecolor='black', linewidth=0.4)
for k, (r, p) in enumerate(zip(rho_vals, rho_combined_df['pvalue'].values)):
    if np.isfinite(p) and p < 0.05:
        ax_rho.text(r + (0.04 if r >= 0 else -0.04), k, '*', va='center',
                    ha='left' if r >= 0 else 'right', fontsize=11)
ax_rho.axvline(0, color='0.4', lw=0.8); ax_rho.set_ylim(ng - 0.5, -0.5)
ax_rho.set_yticks(yv); ax_rho.set_yticklabels(union_genes, fontsize=7)
ax_rho.set_xlim(-1, 1); ax_rho.set_xlabel('Spearman ρ (plasma vs tumour)', fontsize=8)
ax_rho.set_title('Per-gene concordance', fontsize=10, fontweight='bold'); ax_rho.grid(axis='x', alpha=0.25)

cb = fig.colorbar(im, ax=ax_rho, location='left', pad=0.32, fraction=0.05)
cb.set_label('row z-score', fontsize=8); cb.ax.tick_params(labelsize=7)

fig.legend(handles=[Patch(facecolor=PLAT_COLORS['FF'],   label=f'FF (n={len(P_ff)})'),
                    Patch(facecolor=PLAT_COLORS['FFPE'], label=f'FFPE (n={len(P_ffpe)})'),
                    Patch(facecolor='#c44e52', label='positive ρ'),
                    Patch(facecolor='#4c78a8', label='negative ρ')],
           loc='lower center', ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.02))
save_panel(fig, 'supp_fig_s7_paired_rna_protein', copy=True)

## Save tables

In [ ]:
sig_corr_df.to_csv(OUTROOT / 'signature_gene_correlations.csv', index=False)
conc_df.to_csv(OUTROOT / 'signature_concordance_rho.csv', index=False)
deconv_corr_df.to_csv(OUTROOT / 'deconvolution_vs_plasma_score.csv', index=False)

## 8. Summary

In [ ]:
print(f'Matched patients: {len(ff_ids)} FF + {len(ffpe_ids)} FFPE = {len(ff_ids)+len(ffpe_ids)}')
print(f'  FF:   {(ff_groups=="Inflamed").sum()} Inflamed, {(ff_groups=="Non-inflamed").sum()} Non-inflamed')
print(f'  FFPE: {(ffpe_groups=="Inflamed").sum()} Inflamed, {(ffpe_groups=="Non-inflamed").sum()} Non-inflamed')

print('\nPer-gene signature correlation (plasma NPX vs tumour VST):')
for tissue in ['FF', 'FFPE']:
    sub = sig_corr_df[sig_corr_df['tissue'] == tissue]
    print(f'   {tissue}: median \u03c1 = {sub["rho"].median():.3f}, range [{sub["rho"].min():.2f}, {sub["rho"].max():.2f}]')
    for _, r in sub[sub['pvalue'] < 0.05].iterrows():
        print(f'     {r["gene"]}: \u03c1={r["rho"]:.2f}, P={r["pvalue"]:.3g}')

print('\nSignature-level concordance:')
for _, r in conc_df.iterrows():
    print(f'   {r["signature"]} ({r["tissue"]}): \u03c1={r["rho"]:.2f}, P={r["pvalue"]:.3g}, n={r["n"]}')

print('\nTumour immune cells vs plasma Integrated Immune score (P < 0.1):')
for tissue in ['FF', 'FFPE']:
    sub = deconv_corr_df[deconv_corr_df['tissue'] == tissue]
    hits = sub[sub['pvalue'] < 0.1]
    if len(hits):
        for _, r in hits.iterrows():
            print(f'   {tissue}: {r["cell_type"]} \u03c1={r["rho"]:.2f}, P={r["pvalue"]:.3g}')
    else:
        print(f'   {tissue}: none')

## 9. Saved outputs

In [ ]:
print(f'Figures in {PANEL_V2}:')
for _p in sorted(PANEL_V2.glob('*paired_rna*')):
    print('  ', _p.name)
print(f'\nTables in {OUTROOT}:')
for _p in sorted(OUTROOT.glob('*.csv')):
    print('  ', _p.name)